# Langgraph memory 
- checkpoint 
    - graph execution state 
    - conversation continuity 
    - thread-specific 
- store 
    - saves long-term memories 
    - shared scross thread/sessions 
    - user specific or application specific 

# checkpoint 
- messages 
- current state 
- intermediate outputs 
- tool results 
- next node to execute 

In [1]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from typing import TypedDict 


checkpoint=InMemorySaver()
store=InMemoryStore()

class State(TypedDict):
    messages: list 

def chatbot(state):
    messages=state["messages"]
    user_message=messages[-1]

    response=f'you said {user_message}'

    return {
        "messages":messages+[response]
    }

builder=StateGraph(State)
builder.add_node("chatbot",chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot",END)
memory=MemorySaver()
graph=builder.compile(checkpointer=memory)


In [3]:
config = {
    "configurable": {
        "thread_id": "user123"
    }
}

graph.invoke(
    {"messages": ["Hello"]},
    config=config
)


{'messages': ['Hello', 'you said Hello']}

In [4]:
graph.invoke(
    {"messages": ["What did I say?"]},
    config=config
)

{'messages': ['What did I say?', 'you said What did I say?']}

In [5]:
config2 = {
    "configurable": {
        "thread_id": "user456"
    }
}

graph.invoke(
    {"messages": ["What did I say?"]},
    config=config2
)

{'messages': ['What did I say?', 'you said What did I say?']}